In [ ]:
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 1
%aimport utility.functions
%aimport src.analysis
%aimport src.dataset
%aimport src.plots
%aimport src.simulation
import numpy as np
import src.analysis as an
import src.dataset as ds
import torch
from scipy.stats import norm
from utility.double_beta_spectrum import pdf_ratio2b
import utility.functions as fn
BASE = "../"
channels = [3, 5, 9, 11, 13, 15, 17, 19]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]
StdCuts = [0.0003, 0.0004, 0.0008, 0.0006, 0.0012, 0.0007, 0.00035, 0.0025]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
meas_name = "000813_20230628T161508"
channel = 5
sampling_rate = 2000
signal_amp = 11.182/1000
acceptance = 0.9
N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100) #number of sigma corresponding to the desired acceptance
#define the time and ratio grid for the analysis
t_min, t_max, N_t = 0, 8e-4, 100
r_min, r_max, N_r = 0., .5, 100
#get the corresponding ratio distribution 
ratio_distribution = pdf_ratio2b(np.linspace(r_min, r_max, N_r))
ratio_distribution /= np.mean(ratio_distribution)

window_size = 800 #size of the window used for the analysis, should be the same as the one used for the NPS computation
n_trials=200 #number of trials for the optimization of the filters

In [ ]:
meanpulse = np.fromfile(f"/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin_edmean.bin")
rawdata_path = f"/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin"
dataset = ds.CachedBinaryDataset(rawdata_path, window_size,np.arange(0,20000)*window_size)
std= dataset.data.std(dim=1)
nps = an.create_NPS(
    dataset,
    rms_thr=0.00023,
    window_fct = np.hanning,
    batch_size = 2048 )
nps *= (8. / 3.)  # Adjust for Hann window effect

In [ ]:
S, w, H_unit = an.compute_H(meanpulse, nps, np.hanning,sampling_rate=sampling_rate)

In [ ]:
sigma_analytic = an.compute_sigma_OF(S, nps)
print(f"Analytic sigma: {sigma_analytic*1000:.4f} mV")
print(f"signal_amp: {signal_amp*1000:.4f} mV")
print(f"SNR: {signal_amp / sigma_analytic:.2f}")

In [ ]:
data = dataset.data.cpu().numpy()[std < 0.00023]
data -= np.mean(data, axis=1, keepdims=True)
amp = np.mean(np.fft.fft(data*np.hanning(window_size),axis=1)*H_unit, axis=1).real*1000
_,bins,_=plt.hist(amp, bins=100,label = "Noise Data")
bin_centers = (bins[:-1] + bins[1:]) / 2
plt.plot(bin_centers, norm.pdf(bin_centers, scale=sigma_analytic*1000)*len(amp)*(bins[1]-bins[0]), color="red", label=f"Analytic prediction: $\sigma = {sigma_analytic*1000:.2f}$mV")
plt.legend()
plt.ylabel("Counts")
plt.xlabel("Amplitude (mV)")

In [ ]:
S_torch = torch.tensor(S, dtype=torch.cfloat, device=device)
H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat, device=device)
w_torch = torch.tensor(w, dtype=torch.cfloat, device=device)
nps_torch = torch.tensor(nps, dtype=torch.cfloat, device=device)
signal_amp_torch = torch.tensor(signal_amp, dtype=torch.float32, device=device)
t_torch = torch.linspace(t_min, t_max, N_t, dtype=torch.cfloat, device=device)
r_torch = torch.linspace(r_min, r_max, N_r, dtype=torch.cfloat, device=device)
ratio_distribution_torch = torch.tensor(ratio_distribution, dtype=torch.cfloat, device=device)

In [ ]:
f1_t, f2_t, J_values = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch,
                                           signal_amp_torch, ratio_distribution_torch, N_sigma=N_sigma, 
                                           activation_fct = torch.abs,
                                           f1_init = None, f2_init = None,
                                           n_trials=300, use_interp = True, verbose=True)


In [ ]:
f1_opt,f2_opt = f1_t.cpu().numpy(), f2_t.cpu().numpy()
np.save(f"{BASE}/outputs/training_Js/channel_{channel}/J_{int(acceptance * 100):d}_RUN14_meas204", np.array(J_values))
np.save(f"{BASE}/outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance * 100):d}_RUN14_meas204",np.array([f1_opt, f2_opt]))
BI_estimate = J_values[-1] * fn.K #Here K is a constant that depends on crystal size, Mo-2b2n decay rate, maximum delay studied,...
BI_estimate

In [ ]:
plt.loglog(f1_t[:window_size//2], label="f1")
plt.loglog(f2_t[:window_size//2], label="f1")
#plt.loglog(f1_opt[:window_size//2], label="f1")
#plt.loglog(f2_opt[:window_size//2], label="f1")

In [ ]:
meanpulse_inj = np.fromfile(f"/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin_edmean.bin")


rawdata_path = f"/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin"
path_pos_single ="/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin_pos_single.pos"
pos_file = np.loadtxt(path_pos_single)
print(np.mean(pos_file[:, 1]))
data_single = ds.CachedBinaryDataset_withgenerated(rawdata_path, path_pos_single, window_size, pulse=meanpulse_inj, n_windows=15999, win_shift=0)
path_pos_pileup ="/local/home/mp274748/Documents/data/RUN14/000204_20260405T115934_071_000.bin_pos_pileup.pos"
data_pileup = ds.CachedBinaryDataset_withgenerated(rawdata_path, path_pos_pileup, window_size, pulse=meanpulse_inj, n_windows=15999, win_shift=0)


In [ ]:
BI, rp, PSD_pileup, PSD_single, Amp_1_pileup, Amp_2_pileup, Amp_1_single, Amp_2_single = an.compute_BI_torch(
            data_pileup,
            data_single,
            acceptance,
            H_unit_torch,
            f1_t,
            f2_t,
            window_fct = np.hanning,
            compute_uncertainty = False,
            batch_size = 2048,
            use_loader = False,
            full_output = True
        )
print(BI*1e5, rp)
plt.hist(PSD_single, bins=100,range=(0.85,1.02), alpha=0.5, label="Single pulse", color="red")
plt.hist(PSD_pileup, bins=100,range=(0.85,1.02), alpha=0.5, label="Pileup pulse", color="blue")
cut = np.percentile(PSD_single,(1-acceptance)*100)
plt.axvline(cut, color="black", linestyle="--", label="Threshold")